# MovieLens — Preprocessing

This notebook handles data loading, cleaning, and preparation for the MovieLens analysis.


In [ ]:
import pandas as pd
import numpy as np
import itertools

## Load Raw Data


In [ ]:
# Load human annotations
human = pd.read_csv("../data/human_annotations.csv")

# Basic EDA
print("=" * 50)
print("HUMAN ANNOTATIONS")
print("=" * 50)
print(f"Missing values: {human.isnull().sum().sum()}")
print(f"Shape: {human.shape}")
print(f"Unique item IDs: {human['item_id'].nunique()}")
print()
print("Columns:", list(human.columns))
human.head()


In [ ]:
# Load LLM ratings
llm = pd.read_csv("../data/llm_ratings_gpt-4o-mini.csv")

# Basic EDA
print("=" * 50)
print("LLM RATINGS")
print("=" * 50)
print(f"Missing values: {llm.isnull().sum().sum()}")
print(f"Shape: {llm.shape}")
print(f"Unique item IDs: {llm['item_id'].nunique()}")
print()
print("Columns:", list(llm.columns))
llm.head()


## Data Cleaning


In [ ]:
# Rename columns for clarity
human.columns = ["item_id"] + [f"human_#{i}" for i in range(1, human.shape[1])]
print("Human columns renamed to:", list(human.columns))

llm.columns = ["item_id", "GPT-4o-mini"]
print("LLM columns renamed to:", list(llm.columns))

In [ ]:
# Merge datasets
merged = pd.merge(human, llm, on="item_id", suffixes=("_human", "_llm"))
print(f"Merged dataset shape: {merged.shape}")
print(f"Columns: {list(merged.columns)}")


## Explore Human Annotator Combinations

Find the best combination of at least 3 humans that provide the most data points.


In [ ]:
# Identify human and LLM columns
human_cols = [c for c in merged.columns if c.startswith("human_")]
llm_col = "GPT-4o-mini"

results = []

# Loop through all combinations of ≥3 humans
for k in range(3, len(human_cols) + 1):
    for combo in itertools.combinations(human_cols, k):
        subset_cols = list(combo) + [llm_col]
        # Drop only NaNs in this subset
        n_valid = merged.dropna(subset=subset_cols).shape[0]
        results.append({
            "num_humans": k,
            "combination": combo,
            "n_items": n_valid
        })

# Convert to DataFrame and sort by n_items
results_df = pd.DataFrame(results).sort_values("n_items", ascending=False)

# Display top results
print("🔝 Top combinations (≥3 humans) with most complete rows:")
print(f"Results DataFrame shape: {results_df.shape}")
print()
results_df.head(10)


## Save Cleaned Data


In [ ]:
# Save cleaned human annotations
human.to_csv("../data/human_annotations_cleaned.csv", index=False)
print(f"✅ Saved human annotations to ../data/human_annotations_cleaned.csv")
print(f"   Shape: {human.shape}")

# Save cleaned LLM ratings
llm.to_csv("../data/llm_ratings_gpt-4o-mini_cleaned.csv", index=False)
print(f"✅ Saved LLM ratings to ../data/llm_ratings_gpt-4o-mini_cleaned.csv")
print(f"   Shape: {llm.shape}")

# Save merged dataset
merged.to_csv("../data/merged_cleaned.csv", index=False)
print(f"✅ Saved merged dataset to ../data/merged_cleaned.csv")
print(f"   Shape: {merged.shape}")
